# DCO phase-noise shape calculations

This code is designed for the DCO phase noise shaping.

It can support extracting from simulations, or deducing the paper one.

First: Initializations.

In [8]:
# Libraries needed
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

freqs = np.logspace(4, 7.8, 500)
omega = 2j * np.pi * freqs

! mkdir -p outputs

version = "tsmcn28"  # Assign it to "paper", "tsmcn65"

## Check the DCO Path Noise Model

A direct comparison between the transistor simulation and the model

In [ ]:
# Run the simulation if exists
do_sim = version != "paper"
done_sim = False

project_path = "pll_gen"
if version == "tsmcn28":
    project_path = "pll_gen_28nm"

if not os.path.exists(f'../../{project_path}/sim/outputs/test_dco_noise.printsnpn0') and do_sim:
    if os.path.exists(f'../../{project_path}/sim/Makefile'):
        os.system(f"make -C ../../{project_path}/sim/ test_dco_noise")
elif do_sim:
    done_sim = True  # Already done

# Extract the simulation data if existed
if done_sim:
    df = pd.read_csv(
        '../sim/outputs/test_dco_noise.printsnpn0',      # your filename
        delim_whitespace=True,
        comment='#',         # if lines start with # or some header you want to skip
        header=1,            # if there is no well-formatted header row
        names=['HERTZ', 'PAC_L(f)']
    )
    sim_dco_f = df['HERTZ'].to_numpy()
    sim_dco_pn = df['PAC_L(f)'].to_numpy()

    # Extract the -30db and -20db
    sim_dco_fdec = np.log10(sim_dco_f)
    sim_dco_pn_deriv = np.diff(sim_dco_pn)/np.diff(sim_dco_fdec)
    sim_dco_pn_deriv2 = np.diff(sim_dco_pn_deriv)/np.diff(sim_dco_fdec[:-1])

    # We detect the flicker area of -30db by the first derivative
    flicker_area_samples = np.where(sim_dco_pn_deriv <= -30)[0]
    flicker_a = flicker_area_samples[0]
    flicker_b = flicker_area_samples[-1]
    flicker_pseudo_m = -30  # The 30 is just the -30db, and calculate b by calculate mean(yp - m * xp)
    flicker_pseudo_b = np.mean(sim_dco_pn[flicker_a:flicker_b] - flicker_pseudo_m * sim_dco_fdec[flicker_a:flicker_b])

    # Quickly draw the derivative before
    plt.figure(figsize=(6, 8))
    fig, ax = plt.subplots(2, 1, sharex=True)
    ax[0].semilogx(sim_dco_f, sim_dco_pn, color="#9C9C9C", label="DCO (sim)")
    ax[0].set_title('DCO Phase Noise - From Simulations')
    ax[0].set_ylabel('Magnitude (dB)')
    ax[0].grid(True, which="both", ls="-")
    ax[0].legend(fontsize=8, loc='lower left')

    ax[1].semilogx(sim_dco_f[:-2], sim_dco_pn_deriv2, label="DCO pn 2nd deriv")
    ax[1].semilogx(sim_dco_f[:-1], sim_dco_pn_deriv, label="DCO pn 1st deriv")
    ax[1].set_title('Change of DCO phase noise from simulations')
    ax[1].set_ylabel('Magnitude change ($delta$dB)')
    ax[1].set_xlabel('Frequency (Hz)')
    #ax[1].xlabel('Frequency decade (10^x Hz)')
    ax[1].grid(True, which="both", ls="-")
    ax[1].legend(fontsize=8, loc='upper left')

    fig.set_size_inches(6, 8)
    fig.savefig("outputs/sim_DCO_pn_deriv.pdf")
    fig.tight_layout()
    plt.show()

    thermal_deriv_cross = np.where(sim_dco_pn_deriv > -20)[0][0]  # If this fails, it means there is no cross with 20db. Check the above figure
    thermal_deriv2_max_ind = np.argmax(sim_dco_pn_deriv2[:thermal_deriv_cross])  # Get the local maximum of the second deriv
    thermal_deriv2_max = sim_dco_pn_deriv2[thermal_deriv2_max_ind]
    thermal_deriv2_min_ind = np.argmin(sim_dco_pn_deriv2[thermal_deriv2_max_ind:]) + thermal_deriv2_max_ind  # Get the local maximum of the second deriv
    thermal_deriv2_min = sim_dco_pn_deriv2[thermal_deriv2_min_ind]
    thermal_deriv2_max = thermal_deriv2_min + (thermal_deriv2_max-thermal_deriv2_min) * 0.8  # Re-adjust the search
    thermal_area_samples = np.where(np.bitwise_and(
        thermal_deriv2_min <= sim_dco_pn_deriv2[thermal_deriv2_max_ind:], 
        sim_dco_pn_deriv2[thermal_deriv2_max_ind:] <= thermal_deriv2_max))[0]
    thermal_a = thermal_area_samples[0] + thermal_deriv2_max_ind
    thermal_b = thermal_area_samples[-1] + thermal_deriv2_max_ind
    thermal_pseudo_m = -20  # The 20 is just the -30db, and calculate b by calculate mean(yp - m * xp)
    thermal_pseudo_b = np.mean(sim_dco_pn[thermal_a:thermal_b] - thermal_pseudo_m * sim_dco_fdec[thermal_a:thermal_b])

    # Find the flicker corner in the line crossing
    fkr_cnr_dec = (thermal_pseudo_b - flicker_pseudo_b) / (flicker_pseudo_m - thermal_pseudo_m)
    fkr_cnr = np.power(10, fkr_cnr_dec)

    # Get the K_VA from the inflection point
    K_WA_eval_f = fkr_cnr
    K_WA_eval_pn_db = np.interp(K_WA_eval_f, sim_dco_f, sim_dco_pn)
    K_WA_eval_pn = np.power(10, K_WA_eval_pn_db/10)
    K_WA = K_WA_eval_pn * np.power(fkr_cnr, 2) / 2

else:
    # Extract the constants according to the original paper (figure 2)
    # NOTE: Those are highly manual. You are free to just write whatever you want
    flicker_pseudo_m = -30  # The 30 is just the -30db, and calculate b by calculate mean(yp - m * xp)
    thermal_pseudo_m = -20  # The 20 is just the -20db, and calculate b by calculate mean(yp - m * xp)
    sim_dco_f = freqs.copy()  # np.logspace(4, 8, 1000)
    sim_dco_fdec = np.log10(sim_dco_f)

    # Flicker point a = 110e3 [Hz], -78 [db]
    flicker_a_f = 110e3
    flicker_a_pn_db = -78
    
    flicker_pseudo_b = flicker_a_pn_db - flicker_pseudo_m * np.log10(flicker_a_f)

    # Thermal point a = 10e6 [Hz], -125 [db]
    thermal_a_f = 10e6
    thermal_a_pn_db = -125
    thermal_pseudo_b = thermal_a_pn_db - thermal_pseudo_m * np.log10(thermal_a_f)

    # Find the flicker corner in the line crossing
    fkr_cnr_dec = (thermal_pseudo_b - flicker_pseudo_b) / (flicker_pseudo_m - thermal_pseudo_m)
    fkr_cnr = np.power(10, fkr_cnr_dec)

    # Get the K_VA (Thermal noise factor) by using any random value in the thermal area
    # I think it can work also in the flicker area
    K_WA_eval_f = thermal_a_f
    K_WA_eval_pn = np.power(10, thermal_a_pn_db/10)
    # K_WA = K_WA_eval_pn * np.power(fkr_cnr, 2) / 2
    K_WA = K_WA_eval_pn / (1/np.power(K_WA_eval_f, 2) + fkr_cnr/np.power(K_WA_eval_f, 3))

Kfkr_WA = fkr_cnr * K_WA  # Get the flicker noise factor
n_psd_vcoa = Kfkr_WA / np.power(freqs, 3) + K_WA / np.power(freqs, 2)  # Derived VCO spectrum
print(f"fkr_cnr = {fkr_cnr}, K_WA = {K_WA}, Kfkr_WA = {Kfkr_WA}")

flicker_disp_y = sim_dco_fdec*flicker_pseudo_m + flicker_pseudo_b
thermal_disp_y = sim_dco_fdec*thermal_pseudo_m + thermal_pseudo_b

plt.figure(figsize=(6, 4))
if done_sim:
    plt.semilogx(sim_dco_f, sim_dco_pn, color="#000000", label=f"DCO PN (sim)")
plt.semilogx(freqs, 10 * np.log10(n_psd_vcoa), color="#ff7575", label="DCO PN (derived)")
plt.semilogx(sim_dco_f, flicker_disp_y, color="#1f6829", label="Flicker slope approx")
plt.semilogx(sim_dco_f, thermal_disp_y, color="#1f2468", label="Thermal slope approx")

plt.title('DCO phase noise - SIM vs MODEL')
plt.ylabel('Magnitude (dB)')
plt.xlabel('Frequency (Hz)')
plt.grid(True, which="both", ls="-")
plt.legend(fontsize=8, loc='lower left')
plt.tight_layout()
plt.savefig("outputs/dco_cmp.pdf")
plt.show()